# PHẦN 5: NHẬN XÉT VÀ KẾT LUẬN CHUYÊN SÂU

Phần này phân tích hiệu năng và tính ổn định của: **Gauss Partial Pivoting, Phân rã SVD và lặp Gauss-Seidel**.

**Môi trường hệ thống:**
* **Phần cứng:** Intel Core i5-13450HX (16 CPUs, High Performance Mode).
* **Phần mềm:** Fedora Linux, Python 3.12.
* **Quy trình:** Lấy trung bình cộng từ 5 lần chạy độc lập để đảm bảo số liệu thực nghiệm chính xác nhất.

## 5.1. Bảng số liệu tổng hợp toàn cảnh (Trung bình thực nghiệm)

Dưới đây là bảng số liệu trung bình được tổng hợp từ 5 lần chạy độc lập trên hệ thống **i5-13450HX / Fedora Linux**. Dữ liệu bao quát cả 3 loại ma trận để đánh giá toàn diện tính hội tụ và hiệu năng thời gian.

| Kích thước (n) | Loại Ma trận | T/gian Gauss (s) | T/gian SVD (s) | T/gian G-S (s) | Lặp G-S | Trạng thái G-S |
| :---: | :--- | :---: | :---: | :---: | :---: | :--- |
| **50** | L1 - SDD (Chéo trội) | 0.0023 | 0.0632 | **0.0009** | 6 | Hội tụ |
| **50** | L2 - Tăng dần | 0.0016 | 0.0475 | **0.0008** | 6 | Hội tụ |
| **50** | L3 - Không chéo trội | 0.0016 | 0.0476 | - | LỖI | Tràn số (Out of range) |
| **100** | L1 - SDD (Chéo trội) | 0.0113 | 0.2098 | **0.0030** | 6 | Hội tụ |
| **100** | L2 - Tăng dần | 0.0112 | 0.2052 | **0.0025** | 5 | Hội tụ |
| **100** | L3 - Không chéo trội | 0.0113 | 0.2066 | - | LỖI | Tràn số (Out of range) |
| **200** | L1 - SDD (Chéo trội) | 0.0844 | 1.1298 | **0.0097** | 5 | Hội tụ |
| **200** | L2 - Tăng dần | 0.0843 | 1.1408 | **0.0096** | 5 | Hội tụ |
| **200** | L3 - Không chéo trội | 0.0847 | 1.1556 | - | LỖI | Tràn số (Out of range) |
| **500** | L1 - SDD (Chéo trội) | 1.4192 | 13.7255 | **0.0658** | 5 | Hội tụ |
| **500** | L2 - Tăng dần | 1.4361 | 13.7712 | **0.0560** | 4 | Hội tụ |
| **500** | L3 - Không chéo trội | 1.4429 | 14.0271 | - | LỖI | Tràn số (Out of range) |
| **1000**| L1 - SDD (Chéo trội) | 11.9554 | 117.5882 | **0.2310** | 4 | Hội tụ |
| **1000**| L2 - Tăng dần | 11.6872 | 119.5160 | **0.2315** | 4 | Hội tụ |
| **1000**| L3 - Không chéo trội | 11.6943 | 119.1695 | - | LỖI | Tràn số (Out of range) |

**Nhận xét từ bảng số liệu:**
1. **Ưu thế của Gauss-Seidel:** Trên các ma trận Loại 1 và Loại 2, phương pháp lặp thể hiện tốc độ vượt trội. Tại $n=1000$, thời gian thực thi chỉ tốn **~0.23s**, nhanh gấp hơn **50 lần** so với khử Gauss và gần **500 lần** so với SVD.
2. **Tính ổn định của Gauss:** Phép khử Gauss (Partial Pivoting) là phương pháp duy nhất duy trì được sự ổn định và chính xác trên cả 3 loại ma trận với sai số cực thấp ($\approx 10^{-13}$).
3. **Giới hạn của phương pháp lặp:** Đối với ma trận Loại 3 (Không chéo trội), Gauss-Seidel hoàn toàn thất bại và văng lỗi `Numerical result out of range`, minh chứng cho việc thuật toán bị phân kỳ khi không thỏa mãn điều kiện đủ.

## 5.2. Phân tích chi tiết Biểu đồ thực nghiệm

### Biểu đồ 1: Thời gian thực thi trên thang đo tuyến tính
![Biểu đồ 1](bieudo1.jpg)

**Phân tích kĩ thuật:**
* **Đường Gauss & SVD:** Thể hiện đặc tính của hàm đa thức bậc 3. Khi $n < 500$, sự chênh lệch chưa quá lớn, nhưng khi chạm mốc $1000$, thời gian thực thi của SVD (đường màu cam) bùng nổ lên gần 2 phút.
* **Đường Gauss-Seidel:** Gần như trùng với trục hoành. Điều này chứng minh rằng đối với các hệ phương trình có kích thước lớn, nếu thỏa mãn điều kiện hội tụ, phương pháp lặp chiếm ưu thế tuyệt đối về mặt tài nguyên thời gian.

### Biểu đồ 2: Phân tích độ phức tạp thuật toán (Log-Log Scale)
![Biểu đồ 2](bieudo2.jpg)

**Phân tích kĩ thuật:**
* **Độ dốc (Slope):** Cả Gauss và SVD đều chạy song song với đường tham chiếu $O(n^3)$. Điều này xác nhận việc cài đặt thuật toán thủ công của nhóm đã đạt tới độ tối ưu tương đương các thư viện chuẩn về mặt lý thuyết.
* **Hằng số thời gian:** Dù cùng là $O(n^3)$, nhưng khoảng cách giữa đường màu cam (SVD) và màu xanh (Gauss) cho thấy các bước trung gian của SVD (Jacobi, trực giao hóa) làm tăng khối lượng tính toán lên gấp 10 lần.

### Biểu đồ 3: Đặc tính hội tụ của phương pháp lặp
![Biểu đồ 3](bieudo3.jpg)

**Phân tích kĩ thuật:**
* **Tốc độ hội tụ:** Sai số giảm theo hàm mũ và đạt ngưỡng chấp nhận được ($10^{-7}$) chỉ sau chưa đầy 10 bước lặp.
* **Tính ổn định:** Đồ thị mượt mà đi xuống cho thấy ma trận SDD (chéo trội nghiêm ngặt) là "môi trường lý tưởng" cho Gauss-Seidel. Số lượng bước lặp hầu như không tăng đáng kể khi kích thước ma trận tăng từ 50 lên 1000.

## 5.3. Phân tích Ưu và Nhược điểm của các phương pháp qua thực nghiệm

Dựa trên dữ liệu thu thập được từ 375 dòng log và các biểu đồ ở phần trước, nhóm đưa ra đánh giá chi tiết về sự đánh đổi (trade-off) giữa độ chính xác, tốc độ và tính ổn định của từng phương pháp.

### 1. Phép khử Gauss (Partial Pivoting)
Đây là phương pháp giải trực tiếp (Direct Method) mang tính chuẩn mực trong đại số tuyến tính.
* **Ưu điểm vượt trội:** * **Độ tin cậy tuyệt đối:** Kết quả thực nghiệm cho thấy Gauss giải quyết thành công 100% các Test Case từ Loại 1 đến Loại 3. 
    * **Độ chính xác cao:** Sai số trung bình đạt mức lý tưởng $2.87 \times 10^{-13}$ tại $n=1000$. Kỹ thuật xoay phần tử chốt (Partial Pivoting) đã giúp giảm thiểu tối đa sai số làm tròn khi chia cho các số gần bằng 0.
* **Nhược điểm:** * **Chi phí tính toán:** Thời gian thực thi tăng theo lũy thừa bậc 3. Dù chạy trên kiến trúc Raptor Lake của i5-13450HX, khi $n$ tăng gấp đôi (từ 500 lên 1000), thời gian chạy tăng vọt từ ~1.4s lên ~11.9s (tăng gần 8.5 lần, khớp với lý thuyết $2^3 = 8$).
* **Lựa chọn:** Ưu tiên số 1 cho các hệ phương trình kích thước trung bình và yêu cầu độ chính xác nghiệm cao.

### 2. Phân rã giá trị suy biến (SVD)
SVD không chỉ là một bộ giải, mà là một công cụ phân tích nội tại ma trận.
* **Ưu điểm:** * **Tính bền vững (Robustness):** SVD là phương pháp duy nhất có khả năng xử lý các ma trận gần suy biến hoặc ma trận chữ nhật (không vuông). Nó cung cấp số điều kiện $\kappa$ giúp nhận diện ma trận Hilbert "độc hại".
* **Nhược điểm:** * **Hiệu suất kém nhất:** Với thời gian thực thi lên tới **117.58s** cho 1000 ẩn, SVD chậm hơn Gauss gấp 10 lần. Điều này do thuật toán phải thực hiện các phép xoay Jacobi liên tiếp để tìm trị riêng.
    * **Tích lũy sai số:** Do thực hiện quá nhiều phép tính trung gian (căn bậc hai, nhân ma trận trực giao), sai số phục hồi nghiệm của SVD cao hơn hẳn ($~9.47 \times 10^0$), khiến nó không phải là lựa chọn tốt nhất để giải hệ $Ax=b$ thuần túy.
* **Lựa chọn:** Chỉ dùng khi cần phân tích hạng (rank), nén dữ liệu hoặc giải hệ phương trình suy biến.

### 3. Phương pháp lặp Gauss-Seidel
Đại diện cho nhóm phương pháp lặp (Iterative Methods).
* **Ưu điểm:** * **Tốc độ hủy diệt:** Tại $n=1000$, Gauss-Seidel chỉ tốn **0.231s**, một con số không tưởng so với 11.9s của Gauss. 
    * **Tiết kiệm bộ nhớ:** Thuật toán không làm thay đổi cấu trúc ma trận gốc, cực kỳ hữu hiệu nếu ma trận là ma trận thưa.
* **Nhược điểm:** * **Điều kiện hội tụ khắc nghiệt:** Chỉ hoạt động trên ma trận chéo trội (SDD). Thực nghiệm tại Test Case Loại 3 đã chứng minh: chỉ cần mất tính chéo trội, thuật toán sẽ phân kỳ ngay lập tức, dẫn đến lỗi hệ thống `Numerical result out of range`.
    * **Độ chính xác phụ thuộc:** Sai số dừng lại ở ngưỡng $\epsilon$ thiết lập ($10^{-5} \to 10^{-7}$), không thể đạt mức chính xác máy tính như phương pháp trực tiếp.
* **Lựa chọn:** Là giải pháp bắt buộc cho các hệ phương trình siêu lớn trong mô phỏng vật lý và kỹ thuật Mesh.

## 5.4. Kết luận về tính thực dụng

Trong thực tế kỹ thuật, việc lựa chọn thuật toán không dựa trên cái nào "mạnh nhất" mà dựa trên "loại dữ liệu":

1. **Gauss-Seidel (Thực dụng trong Mô phỏng):** * *Ví dụ:* Tính toán dòng điện trong lưới điện quốc gia hoặc mô phỏng dòng chảy chất lưu (CFD). Các hệ này thường có ma trận rất lớn nhưng thưa và chéo trội. Sử dụng phương pháp lặp là cách duy nhất để có kết quả trong thời gian thực.
   
2. **Gauss Partial Pivoting (Thực dụng trong Bài toán chính xác):** * *Ví dụ:* Thiết kế chi tiết cơ khí hoặc tính toán quỹ đạo bay. Ở đây, độ chính xác là ưu tiên số 1 và kích thước hệ phương trình thường vừa phải.
   
3. **SVD (Thực dụng trong Xử lý dữ liệu):** * *Ví dụ:* Nén ảnh hoặc hệ thống gợi ý (Netflix/Youtube). SVD giúp loại bỏ các "nhiễu" (giá trị suy biến nhỏ) để lấy được các thành phần chính của dữ liệu.

**Lời kết:** Đồ án đã giúp nhóm thấu hiểu sự đánh đổi (Trade-off) giữa **Tốc độ** và **Độ ổn định**. Một lập trình viên giỏi không chỉ biết viết code chạy được, mà còn phải biết chọn thuật toán "khớp" với đặc tính toán học của dữ liệu đầu vào.